In [57]:
import numpy as np
import scipy.stats as stats
import heapq

In [58]:
def confidence_interval(samples, confidence=0.95):
    samples = np.asarray(samples)
    n = samples.size
    mean = samples.mean()
    se = samples.std(ddof=1) / np.sqrt(n)

    alpha = 1 - confidence
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)

    theta_hat, ci_lower, ci_upper = mean, mean - t_crit * se, mean + t_crit * se

    return theta_hat, ci_lower, ci_upper

ARRIVAL, DEPARTURE = 0, 1

def simulate_blocking_cv(m, mean_service_time, mean_interarrival_time, num_customers):
    busy_servers = 0
    blocked = 0
    arrived = 0
    accepted = 0
    sum_interarrival = 0.0
    sum_service = 0.0

    events = []
    ia0 = np.random.exponential(mean_interarrival_time)
    sum_interarrival += ia0
    heapq.heappush(events, (ia0, ARRIVAL))

    while arrived < num_customers:
        time, etype = heapq.heappop(events)
        if etype == ARRIVAL:
            arrived += 1
            if busy_servers < m:
                busy_servers += 1
                accepted += 1
                s = np.random.exponential(mean_service_time)
                sum_service += s
                heapq.heappush(events, (time + s, DEPARTURE))
            else:
                blocked += 1
            if arrived < num_customers:
                ia = np.random.exponential(mean_interarrival_time)
                sum_interarrival += ia
                heapq.heappush(events, (time + ia, ARRIVAL))
        else:
            busy_servers -= 1

    blocking_fraction = blocked / num_customers
    mean_interarrival_realized = sum_interarrival / num_customers
    mean_service_realized = sum_service / accepted
    return blocking_fraction, mean_interarrival_realized, mean_service_realized

In [59]:
# Ex 1
np.random.seed(42)

n = 100

U = np.random.uniform(0, 1, size=n)

samples = np.exp(U)

theta_hat, ci_lower, ci_upper = confidence_interval(samples)

print(f"Point estimate: {theta_hat:.4f}")
print(f"Sample std dev: {samples.std(ddof=1):.4f}")
print(f"95% CI:         ({ci_lower:.4f}, {ci_upper:.4f})")


Point estimate: 1.6721
Sample std dev: 0.5001
95% CI:         (1.5728, 1.7713)


In [60]:
# Ex 2
np.random.seed(42)

n = 100

U = np.random.uniform(0, 1, size=n)

xi = np.exp(U)
xiA= np.exp(1-U)
Yi = (xi+xiA)/2

theta_hat=1/n * Yi.sum()

theta_hat, ci_lower, ci_upper = confidence_interval(Yi)

print(f"Point estimate: {theta_hat:.4f}")
print(f"Sample std dev: {Yi.std(ddof=1):.4f}")
print(f"95% CI:         ({ci_lower:.4f}, {ci_upper:.4f})")


Point estimate: 1.7226
Sample std dev: 0.0626
95% CI:         (1.7102, 1.7350)


In [61]:
# Ex 3
np.random.seed(42)

xi = np.exp(U)

Zi = U

c = -(np.cov(xi,Zi)/(Zi.var()))

c = -np.cov(xi, Zi, ddof=1)[0, 1] / Zi.var(ddof=1)
yi = xi + c * (Zi - Zi.mean())

theta_hat, ci_lower, ci_upper = confidence_interval(yi)

print(f"Point estimate: {theta_hat:.4f}")
print(f"Sample std dev: {yi.std(ddof=1):.4f}")
print(f"95% CI:         ({ci_lower:.4f}, {ci_upper:.4f})")


Point estimate: 1.6721
Sample std dev: 0.0621
95% CI:         (1.6597, 1.6844)


In [62]:
# Ex 4
np.random.seed(42)

strata_count = 10
observations = 10

y = np.empty(strata_count)
var = np.empty(strata_count)

strata_lb = np.arange(0, 1, 1 / strata_count)
strata_ub = np.arange(1 / strata_count, 1 + 1 / strata_count, 1 / strata_count)

for i in range(strata_count):
    strata_samples = np.random.uniform(strata_lb[i], strata_ub[i], size=observations)
    strata_exp = np.exp(strata_samples)
    y[i] = strata_exp.mean()
    var[i] = strata_exp.var(ddof=1)

weights = np.full(strata_count, 1 / strata_count)
theta_hat = np.sum(weights * y)

se = np.sqrt(np.sum((weights ** 2) * var / observations))

t_crit = stats.t.ppf(1 - 0.95 / 2, df=strata_count - 1)
ci_lower = theta_hat - t_crit * se
ci_upper = theta_hat + t_crit * se

print(f"Point estimate:      {theta_hat:.4f}")
print(f"Estimator std error: {se:.4f}")
print(f"95% CI:              ({ci_lower:.4f}, {ci_upper:.4f})")

Point estimate:      1.7137
Estimator std error: 0.0054
95% CI:              (1.7133, 1.7140)


In [63]:
# Ex 5
np.random.seed(42)

m = 10
mean_service_time = 8.0
mean_interarrival_time = 1.0
num_customers = 10000
num_runs = 10

X = np.empty(num_runs) 
Z1 = np.empty(num_runs)
Z2 = np.empty(num_runs)

for i in range(num_runs):
    X[i], Z1[i], Z2[i] = simulate_blocking_cv(m, mean_service_time, mean_interarrival_time, num_customers)

theta_crude, ci_lower, ci_upper = confidence_interval(X)

print("Crude estimator (Ex 4):")
print(f"  Point estimate: {theta_crude:.5f}")
print(f"  Sample std dev: {X.std(ddof=1):.5f}")
print(f"  95% CI:         ({ci_lower:.5f}, {ci_upper:.5f})")

Crude estimator (Ex 4):
  Point estimate: 0.11853
  Sample std dev: 0.00600
  95% CI:         (0.11424, 0.12282)


In [64]:
c2 = -np.cov(X, Z2, ddof=1)[0, 1] / Z2.var(ddof=1)
Y2 = X + c2 * (Z2 - mean_service_time)

theta_cv2, ci_lower2, ci_upper2 = confidence_interval(Y2)

print(f"corr(X, Z2) = {np.corrcoef(X, Z2)[0, 1]:.4f}")
print(f"c           = {c2:.4f}\n")
print(f"Point estimate: {theta_cv2:.5f}")
print(f"Sample std dev: {Y2.std(ddof=1):.5f}")
print(f"95% CI:         ({ci_lower2:.5f}, {ci_upper2:.5f})")
print(f"Variance reduction vs crude: {1 - Y2.var(ddof=1) / X.var(ddof=1):.1%}")

corr(X, Z2) = 0.7932
c           = -0.0427

Point estimate: 0.12034
Sample std dev: 0.00365
95% CI:         (0.11772, 0.12295)
Variance reduction vs crude: 62.9%


In [65]:
Zc = np.column_stack([Z1 - mean_interarrival_time, Z2 - mean_service_time])
c_vec = np.linalg.lstsq(Zc, X - X.mean(), rcond=None)[0]
Y3 = X - Zc @ c_vec

theta_cv3, ci_lower3, ci_upper3 = confidence_interval(Y3)

print(f"c = (c1, c2) = ({c_vec[0]:.4f}, {c_vec[1]:.4f})\n")
print(f"Point estimate: {theta_cv3:.5f}")
print(f"Sample std dev: {Y3.std(ddof=1):.5f}")
print(f"95% CI:         ({ci_lower3:.5f}, {ci_upper3:.5f})")
print(f"Variance reduction vs crude: {1 - Y3.var(ddof=1) / X.var(ddof=1):.1%}")
print(f"\nErlang B (analytical) = 0.12166")

c = (c1, c2) = (-0.1625, 0.0358)

Point estimate: 0.12076
Sample std dev: 0.00269
95% CI:         (0.11884, 0.12269)
Variance reduction vs crude: 79.8%

Erlang B (analytical) = 0.12166


In [66]:
# Ex 6:
m_crn, mean_svc_crn, mean_ia_crn = 10, 8.0, 1.0
n_runs_crn, num_cust_crn = 50, 10_000

def simulate_blocking_streams(m, mean_svc, arrival_fn, svc_rng, n_cust):
    busy, blocked, arrived, events = 0, 0, 0, []
    heapq.heappush(events, (arrival_fn(), ARRIVAL))
    while arrived < n_cust:
        t, etype = heapq.heappop(events)
        if etype == ARRIVAL:
            arrived += 1
            if busy < m:
                busy += 1
                heapq.heappush(events, (t + svc_rng.exponential(mean_svc), DEPARTURE))
            else:
                blocked += 1
            if arrived < n_cust:
                heapq.heappush(events, (t + arrival_fn(), ARRIVAL))
        else:
            busy -= 1
    return blocked / n_cust

def make_poisson_fn(rng, mean_ia):
    def _fn(): return rng.exponential(mean_ia)
    return _fn

def make_hyperexp_fn(rng, p1=0.8, lam1=0.8333, lam2=5.0):
    def _fn(): return rng.exponential(1.0/lam1) if rng.random() < p1 else rng.exponential(1.0/lam2)
    return _fn

B_P_ind = np.empty(n_runs_crn)
B_H_ind = np.empty(n_runs_crn)
B_P_crn_arr = np.empty(n_runs_crn)
B_H_crn_arr = np.empty(n_runs_crn)

for i in range(n_runs_crn):
    b = i * 7
    B_P_ind[i] = simulate_blocking_streams(m_crn, mean_svc_crn,
        make_poisson_fn(np.random.RandomState(b+2), mean_ia_crn),
        np.random.RandomState(b), num_cust_crn)
    B_H_ind[i] = simulate_blocking_streams(m_crn, mean_svc_crn,
        make_hyperexp_fn(np.random.RandomState(b+3)),
        np.random.RandomState(b+1), num_cust_crn)

    svc_seed = b + 4
    B_P_crn_arr[i] = simulate_blocking_streams(m_crn, mean_svc_crn,
        make_poisson_fn(np.random.RandomState(b+5), mean_ia_crn),
        np.random.RandomState(svc_seed), num_cust_crn)
    B_H_crn_arr[i] = simulate_blocking_streams(m_crn, mean_svc_crn,
        make_hyperexp_fn(np.random.RandomState(b+6)),
        np.random.RandomState(svc_seed), num_cust_crn)

D_ind = B_P_ind - B_H_ind
D_crn = B_P_crn_arr - B_H_crn_arr
rho = np.corrcoef(B_P_crn_arr, B_H_crn_arr)[0, 1]

print(f"Independent:  mean = {D_ind.mean():.5f},  std of D = {D_ind.std(ddof=1):.5f}")
print(f"CRN:          mean = {D_crn.mean():.5f},  std of D = {D_crn.std(ddof=1):.5f}")
print(f"\nVariance reduction: {1 - D_crn.var(ddof=1) / D_ind.var(ddof=1):.1%}")
print(f"Corr(B_Poisson, B_Hyperexp) under CRN: {rho:.4f}")
print(f"Positive correlation (rho={rho:.3f}) shrinks Var(D), demonstrating CRN's benefit.")

Independent:  mean = -0.01856,  std of D = 0.00890
CRN:          mean = -0.01624,  std of D = 0.00484

Variance reduction: 70.5%
Corr(B_Poisson, B_Hyperexp) under CRN: 0.6717
Positive correlation (rho=0.672) shrinks Var(D), demonstrating CRN's benefit.


In [67]:
# Ex 7
np.random.seed(42)

def true_prob(a):
    return stats.norm.sf(a)

def crude_mc(a, n):
    Z = np.random.normal(0, 1, size=n)
    X = (Z > a).astype(float)
    return confidence_interval(X), X.var(ddof=1)

def importance_sampling(a, sigma2, n):
    sigma = np.sqrt(sigma2)
    Y = np.random.normal(a, sigma, size=n)
    w = sigma * np.exp(-Y**2 / 2 + (Y - a) ** 2 / (2 * sigma2))
    X = (Y > a).astype(float) * w
    return confidence_interval(X), X.var(ddof=1)

n = 10000
for a in (2, 4):
    (theta_mc, lo_mc, hi_mc), var_mc = crude_mc(a, n)
    (theta_is, lo_is, hi_is), var_is = importance_sampling(a, 1.0, n)
    print(f"a = {a}, true P(Z>a) = {true_prob(a):.6}")
    print(f"  Crude MC:    theta_hat = {theta_mc:.6}, var = {var_mc:.3}, 95% CI = ({lo_mc:.3}, {hi_mc:.3})")
    print(f"  IS (sig2=1): theta_hat = {theta_is:.6}, var = {var_is:.3}, 95% CI = ({lo_is:.3}, {hi_is:.3})")
    if var_mc > 0:
        print(f"  Variance reduction: {1 - var_is / var_mc:.4%}")
    else:
        print(f"  Variance reduction: n/a (crude variance = 0, zero hits in {n} draws)")
    print()

a = 2, true P(Z>a) = 0.0227501
  Crude MC:    theta_hat = 0.0237, var = 0.0231, 95% CI = (0.0207, 0.0267)
  IS (sig2=1): theta_hat = 0.0232753, var = 0.00126, 95% CI = (0.0226, 0.024)
  Variance reduction: 94.5584%

a = 4, true P(Z>a) = 3.16712e-05
  Crude MC:    theta_hat = 0.0, var = 0.0, 95% CI = (0.0, 0.0)
  IS (sig2=1): theta_hat = 3.10538e-05, var = 4.4e-09, 95% CI = (2.98e-05, 3.24e-05)
  Variance reduction: n/a (crude variance = 0, zero hits in 10000 draws)



In [68]:
sample_sizes = [100, 1000, 10000, 100000]

for a in (2, 4):
    print(f"a = {a}, true P(Z>a) = {true_prob(a):.6}")
    for n in sample_sizes:
        np.random.seed(42)
        (theta_mc, lo_mc, hi_mc), _ = crude_mc(a, n)
        np.random.seed(42)
        (theta_is, lo_is, hi_is), _ = importance_sampling(a, 1.0, n)
        print(f"  n={n:>7}: crude = {theta_mc:.3} (CI width {hi_mc - lo_mc:.3}),  "
              f"IS = {theta_is:.3} (CI width {hi_is - lo_is:.3})")
    print()

a = 2, true P(Z>a) = 0.0227501
  n=    100: crude = 0.0 (CI width 0.0),  IS = 0.0237 (CI width 0.0143)
  n=   1000: crude = 0.024 (CI width 0.019),  IS = 0.0241 (CI width 0.00445)
  n=  10000: crude = 0.0237 (CI width 0.00596),  IS = 0.0228 (CI width 0.00137)
  n= 100000: crude = 0.0227 (CI width 0.00185),  IS = 0.0228 (CI width 0.000433)

a = 4, true P(Z>a) = 3.16712e-05
  n=    100: crude = 0.0 (CI width 0.0),  IS = 3.38e-05 (CI width 2.69e-05)
  n=   1000: crude = 0.0 (CI width 0.0),  IS = 3.41e-05 (CI width 8.61e-06)
  n=  10000: crude = 0.0 (CI width 0.0),  IS = 3.17e-05 (CI width 2.64e-06)
  n= 100000: crude = 1e-05 (CI width 3.92e-05),  IS = 3.19e-05 (CI width 8.39e-07)



In [69]:
sigma2_values = [0.5, 1, 2, 4, 8]
n = 10000

for a in (2, 4):
    print(f"a = {a}, true P(Z>a) = {true_prob(a):.6e}")
    for sigma2 in sigma2_values:
        np.random.seed(42)
        (theta_is, lo_is, hi_is), var_is = importance_sampling(a, sigma2, n)
        print(f"  sigma^2={sigma2:>4}: theta_hat = {theta_is:.4e}, var = {var_is:.3e}, CI width = {hi_is - lo_is:.3e}")
    print()

a = 2, true P(Z>a) = 2.275013e-02
  sigma^2= 0.5: theta_hat = 2.2735e-02, var = 7.755e-04, CI width = 1.092e-03
  sigma^2=   1: theta_hat = 2.2755e-02, var = 1.214e-03, CI width = 1.366e-03
  sigma^2=   2: theta_hat = 2.2781e-02, var = 1.874e-03, CI width = 1.697e-03
  sigma^2=   4: theta_hat = 2.2788e-02, var = 2.830e-03, CI width = 2.086e-03
  sigma^2=   8: theta_hat = 2.2778e-02, var = 4.207e-03, CI width = 2.543e-03

a = 4, true P(Z>a) = 3.167124e-05
  sigma^2= 0.5: theta_hat = 3.1694e-05, var = 2.971e-09, CI width = 2.137e-06
  sigma^2=   1: theta_hat = 3.1718e-05, var = 4.535e-09, CI width = 2.640e-06
  sigma^2=   2: theta_hat = 3.1725e-05, var = 6.792e-09, CI width = 3.231e-06
  sigma^2=   4: theta_hat = 3.1740e-05, var = 1.005e-08, CI width = 3.929e-06
  sigma^2=   8: theta_hat = 3.1820e-05, var = 1.474e-08, CI width = 4.759e-06



In [70]:
# Ex 8:
from scipy.optimize import minimize_scalar

theta_true = np.e - 1
var_crude = (np.e**2 - 1) / 2 - (np.e - 1)**2

def var_IS_exp(lam):
    return (np.exp(2.0 + lam) - 1.0) / (lam * (2.0 + lam)) - theta_true**2

res = minimize_scalar(var_IS_exp, bounds=(0.01, 20.0), method='bounded')
lam_opt = res.x

print(f"Crude MC variance:          {var_crude:.4f}")
print(f"Analytical optimal lambda:       {lam_opt:.4f}")
print(f"IS variance at optimal lambda:   {var_IS_exp(lam_opt):.4f}  "
      f"({var_IS_exp(lam_opt)/var_crude:.0f} worse than crude MC)")
print()
print("Simulation (n = 10 000):\n")

np.random.seed(42)
n = 10_000
print(f"{'lambda':>6}  {'theta':>10}  {'sample var':>12}  {'CI width':>10}")
for lam in [0.5, lam_opt, 1.0, 2.0, 5.0]:
    Y = np.random.exponential(1.0 / lam, size=n)
    w = np.where(Y <= 1.0, np.exp(Y * (1.0 + lam)) / lam, 0.0)
    est, lo, hi = confidence_interval(w)
    print(f"{lam:>6.2f}  {est:>10.5f}  {w.var(ddof=1):>12.5f}  {hi - lo:>10.5f}")

print(f"\nCrude MC reference: var approximately {var_crude:.4f}")

Crude MC variance:          0.2420
Analytical optimal lambda:       1.3548
IS variance at optimal lambda:   3.1288  (13 worse than crude MC)

Simulation (n = 10 000):

lambda       theta    sample var    CI width
  0.50     1.73099       5.97476     0.09583
  1.35     1.70681       3.12701     0.06933
  1.00     1.73193       3.46537     0.07298
  2.00     1.71072       3.73084     0.07572
  5.00     1.69056      28.74994     0.21021

Crude MC reference: var approximately 0.2420


In [73]:
# Ex 9:
k_p, beta_p = 2.5, 1.0
theta_pareto = beta_p * k_p / (k_p - 1)
np.random.seed(42)
n = 100_000

X_crude = (np.random.pareto(k_p, n) + 1) * beta_p
est_c, lo_c, hi_c = confidence_interval(X_crude)

w_pareto = np.full(n, theta_pareto)

print(f"Pareto(k={k_p}, beta={beta_p}) => E[X] = {theta_pareto:.4f}")
print(f"Crude MC:  mean theta = {est_c:.4f},  std = {X_crude.std(ddof=1):.4f},  CI = ({lo_c:.4f}, {hi_c:.4f})")
print(f"IS (g=g1): mean theta = {w_pareto.mean():.4f},  std = {w_pareto.std(ddof=1):.6f}  (zero variance — exact)")
np.random.seed(42)
n = 10_000
U_opt = np.random.uniform(size=n)
Y_opt = np.log(1.0 + U_opt * (np.e - 1.0))      
w_opt = np.full(n, np.e - 1.0)                   
est_opt, lo_opt, hi_opt = confidence_interval(w_opt)
print(f"mean theta = {est_opt:.5f},  std = {w_opt.std(ddof=1):.6f},  CI = ({lo_opt:.5f}, {hi_opt:.5f})")

Pareto(k=2.5, beta=1.0) => E[X] = 1.6667
Crude MC:  mean theta = 1.6601,  std = 1.3199,  CI = (1.6520, 1.6683)
IS (g=g1): mean theta = 1.6667,  std = 0.000000  (zero variance — exact)
mean theta = 1.71828,  std = 0.000000,  CI = (1.71828, 1.71828)
